## Inicialización de los datos

In [4]:
# Precarga de todas las librerias necesarias para el proyecto

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from collections import Counter
from pathlib import Path

In [5]:
# Definición de las rutas de los archivos CSV a utilizar en el proyecto 

base_dir = Path().resolve()

data_path_train = base_dir / 'data' / 'gold_recovery_train.csv'
data_path_test = base_dir / 'data' / 'gold_recovery_test.csv'  
data_path_full = base_dir / 'data' / 'gold_recovery_full.csv'

# Lectura y almacenamiento en dataframes de los CVS proporcionados 

au_list = {
    'au_train': pd.read_csv(data_path_train),
    'au_test': pd.read_csv(data_path_test),
    'au_full': pd.read_csv(data_path_full) }


In [6]:
# Consulta de datos generales au_train, au_test, au_full

for name, df in au_list.items(): 
    print(f'++ Forma de la tabla {name}: {df.shape} ++\n')
    print(df.info())
    print('\n','-'*100,'\n')


++ Forma de la tabla au_train: (16860, 87) ++

<class 'pandas.DataFrame'>
RangeIndex: 16860 entries, 0 to 16859
Data columns (total 87 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   date                                                16860 non-null  str    
 1   final.output.concentrate_ag                         16788 non-null  float64
 2   final.output.concentrate_pb                         16788 non-null  float64
 3   final.output.concentrate_sol                        16490 non-null  float64
 4   final.output.concentrate_au                         16789 non-null  float64
 5   final.output.recovery                               15339 non-null  float64
 6   final.output.tail_ag                                16794 non-null  float64
 7   final.output.tail_pb                                16677 non-null  float64
 8   final.output.tail_sol                   

In [7]:
# Impresion de muestra de au_train, au_test, au_full

for df in au_list: display(au_list[df].sample(3))

,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
9259,2017-06-05 18:59:59,4.251917,10.636725,8.076898,46.022685,61.144151,8.937325,3.593877,9.165263,3.667957,...,14.974680,-496.825926,9.969688,-379.003796,14.973678,-499.202633,9.993810,-500.127585,14.998421,-500.180405
13156,2018-03-17 03:59:59,4.981612,11.343507,7.925563,44.042980,72.190053,10.007210,2.568092,8.326815,2.762034,...,23.034590,-499.658962,14.964297,-499.602745,18.044661,-499.781130,11.981768,-500.055758,11.978576,-500.129242
6749,2017-02-21 04:59:59,5.813475,10.747704,14.104161,43.976033,73.010765,12.094983,4.189077,8.629817,3.733846,...,24.989349,-399.064774,23.012468,-400.162613,22.998051,-449.256157,19.982860,-449.842581,24.984365,-499.666109


,date,primary_cleaner.input.sulfate,primary_cleaner.input.depressant,primary_cleaner.input.feed_size,primary_cleaner.input.xanthate,primary_cleaner.state.floatbank8_a_air,primary_cleaner.state.floatbank8_a_level,primary_cleaner.state.floatbank8_b_air,primary_cleaner.state.floatbank8_b_level,primary_cleaner.state.floatbank8_c_air,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
4148,2017-10-21 20:59:59,225.958287,13.960422,7.10,1.829708,1699.849346,-499.290150,1700.946299,-486.490115,1654.767808,...,21.000713,-501.216740,19.004196,-401.031767,15.031874,-500.511709,10.980509,-500.103418,16.007900,-500.236458
2003,2016-11-23 11:59:59,196.041197,6.963809,7.38,1.696760,1596.746210,-499.806829,1602.025873,-500.232514,1599.470762,...,18.030640,-501.135997,15.882575,-447.343029,15.395065,-499.946950,11.959839,-500.014192,21.968055,-499.210981
5425,2017-12-14 01:59:59,196.524958,9.989526,8.19,1.100436,1554.171896,-500.106363,1547.485556,-497.016248,1548.260337,...,20.006407,-500.873491,15.048109,-113.572960,10.999067,-499.975324,8.048825,-500.181578,11.990546,-499.904981


,date,final.output.concentrate_ag,final.output.concentrate_pb,final.output.concentrate_sol,final.output.concentrate_au,final.output.recovery,final.output.tail_ag,final.output.tail_pb,final.output.tail_sol,final.output.tail_au,...,secondary_cleaner.state.floatbank4_a_air,secondary_cleaner.state.floatbank4_a_level,secondary_cleaner.state.floatbank4_b_air,secondary_cleaner.state.floatbank4_b_level,secondary_cleaner.state.floatbank5_a_air,secondary_cleaner.state.floatbank5_a_level,secondary_cleaner.state.floatbank5_b_air,secondary_cleaner.state.floatbank5_b_level,secondary_cleaner.state.floatbank6_a_air,secondary_cleaner.state.floatbank6_a_level
19629,2018-04-11 20:59:59,3.542927,9.394940,8.080893,49.219810,72.566337,9.225082,2.123326,7.546866,1.859629,...,23.008370,-501.063222,15.939196,-341.002660,17.952079,-499.945497,12.062322,-499.601582,12.996949,-500.068713
1708,2016-03-26 04:00:00,5.108435,8.708830,1.702668,48.109180,62.287500,10.189546,1.233996,18.874992,2.847869,...,11.956652,-500.525189,11.954754,-500.054336,12.039793,-499.542724,10.021443,-499.873688,19.985900,-500.144534
16858,2017-12-17 09:59:59,7.506471,9.247625,8.265027,28.575838,29.950111,9.377262,1.431288,4.904435,4.025224,...,19.957843,-498.201601,16.074490,-44.244302,11.027198,-497.421379,7.997135,-499.712718,12.001879,-499.602975


In [8]:
# Funcion para limpiar dataframes

def wrangling(data):

    '''
    Consulta la cantidad de celdas vacias y duplicadas del dataframe brindado,
    posteriormente se eliminan las filas con estos datos
    '''

    dup = data.duplicated().sum()
    null = data.isna().sum().sum()
    
    if null > 0: data = data.dropna()

    if dup > 0: data.drop_duplicates().reset_index()

    return data
    

In [9]:
# Comprobacion de datos faltantes y duplicados en au_train, au_test, au_full

print(f"** Dataframes sin limpiar")
for name, df in au_list.items():
    print('-'*40)
    print(name)
    print(f'+ Forma de la tabla: {df.shape}\n+ Duplicados: {df.duplicated().sum()}\n+ Celdas vacias: {df.isna().sum().sum()}\n')
    print('-'*40)

# Llamado de la funcion wrangling para limpiar au_train, au_test, au_full

au_list_cl = {name: wrangling(df) for name, df in au_list.items()}

# Confirmacion de limpieza

print(f"** Dataframes limpios")
for name, df in au_list_cl.items():
    print('-'*40)
    print(name)
    print(f'+ Forma de la tabla: {df.shape}\n+ Duplicados: {df.duplicated().sum()}\n+ Celdas vacias: {df.isna().sum().sum()}\n')
    print('-'*40)

** Dataframes sin limpiar
----------------------------------------
au_train
+ Forma de la tabla: (16860, 87)
+ Duplicados: 0
+ Celdas vacias: 30320

----------------------------------------
----------------------------------------
au_test
+ Forma de la tabla: (5856, 53)
+ Duplicados: 0
+ Celdas vacias: 2360

----------------------------------------
----------------------------------------
au_full
+ Forma de la tabla: (22716, 87)
+ Duplicados: 0
+ Celdas vacias: 36587

----------------------------------------
** Dataframes limpios
----------------------------------------
au_train
+ Forma de la tabla: (11017, 87)
+ Duplicados: 0
+ Celdas vacias: 0

----------------------------------------
----------------------------------------
au_test
+ Forma de la tabla: (5383, 53)
+ Duplicados: 0
+ Celdas vacias: 0

----------------------------------------
----------------------------------------
au_full
+ Forma de la tabla: (16094, 87)
+ Duplicados: 0
+ Celdas vacias: 0

----------------------------